# Week 16 – Research Assistant Agent with Tool Calls 16.3 Graded Mini Project
### Submitted by: Harley Dangan
### Date: 26 November 2026
### Submitted to: The Asian Institute of Management



## Load Environment Variables

This Jupyter notebook assumes that you have the environment variables configured. See .venv.example for reference.
***NOTE: langchain_ollama is used in this notebook. This notebook will only work with Ollama. This gives me unlimited tokens.***

In [8]:
import os

base_url = os.environ.get("BASE_URL")
llm= os.environ.get("LLM")
api_key = os.environ.get("OPENAI_API_KEY") # Actually, any string will work here since I used Ollama!
tavily_api_key = os.environ.get("TAVILY_API_KEY")

if not base_url or not llm or not api_key or not tavily_api_key:
    raise ValueError("Missing required environment variables: BASE_URL, LLM, API_KEY, or TAVILY_API_KEY")
print(f"Base URL: {base_url}")
print(f"LLM: {llm}")
print(f"API Key: {api_key}")
print(f"Tavily API Key: {tavily_api_key}")

Base URL: localhost:11434/v1
LLM: llama-3.2-1b-q4_k_m
API Key: llama
Tavily API Key: your_travily_api_key_here


## Initialize LLM

NOTE: Assumes local model running in Ollama.

In [9]:
from openai import OpenAI
from langchain_ollama import ChatOllama


# Initializes the client
llm = ChatOllama(
        model=llm, 
        temperature=0.3,
        base_url=base_url,
        api_key=api_key # Actually, any string will work here since I used Ollama!
    )

# Defining the system prompt for the model
system_prompt = """You are an expert Research Assistant Agent. Your core objective is to deliver comprehensive, highly accurate, and fact-verified research on any given topic. You have access to tools for searching the web and retrieving full webpage content.

### OPERATIONAL MANDATES
1. FACTUAL GROUNDING: Rely exclusively on verifiable information retrieved from your tools. If a fact cannot be sourced from your tool outputs, explicitly state that it is unavailable. Never assume, extrapolate, or hallucinate details.
2. LATENT SEARCH QUERIES: When an initial tool result is ambiguous, outdated, or incomplete, proactively formulate distinct follow-up queries or browse primary source URLs to fill knowledge gaps.
3. CONTRADICTION RESOLUTION: If different sources present conflicting numbers, dates, or accounts, outline both perspectives objectively. Do not pick a side unless one source is demonstrably more credible.
4. SYNTHESIS OVER SUMMARIZATION: Do not simply list summaries of search results. Synthesize the findings into a cohesive narrative structured with clear headers, logical groupings, and thematic continuity.

### TOOL-CALLING GUIDELINES
- Query Design: Keep search tool queries targeted and optimized. Avoid passing long conversational sentences to search engines. Use precise keywords, dates, or entity names.
- Deep Dive: If a search snippet provides a promising lead but lacks critical context, use the browser tool to read the full page before finalizing your synthesis.
- Temporal Context: Always keep the current year and context in mind to evaluate whether information is current or obsolete.

### OUTPUT STRUCTURE & STYLE
- Formality: Maintain a neutral, professional, and academic tone. Avoid conversational filler, personal pronouns, or fluff.
- Scannability: Use descriptive Markdown headers, bold key entities/metrics, and utilize bullet points for non-sequential data points.
- Citation: Always attach functional anchors or clear indicators mapping back to the specific source domain or URL retrieved by your tools."""

# Uncomment below to test the model with a simple prompt
"""
# Call and test the specified model
print("🤖 Agent analyzing query...")
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

response_message = response.choices[0].message
messages.append(response_message) # Append the model's response to keep history consistent

# Step 5: Check if the model requested a tool call
if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        if tool_call.function.name == "tavily_search":
            # Extract arguments generated by the model
            function_args = json.loads(tool_call.function.allocations or tool_call.function.arguments)
            search_query = function_args.get("query")
            
            print(f"🔍 Agent requested tool: tavily_search(query='{search_query}')")
            
            # Execute the actual web search using Tavily
            tavily_response = tavily_client.search(query=search_query, max_results=3)
            
            # Format results concisely for the LLM
            search_results_str = json.dumps(tavily_response.get("results", []))
            
            # Append the tool result to the conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "tavily_search",
                "content": search_results_str
            })
            
    # Step 6: Second LLM call - Send tool results back to get final answer
    print("🤖 Agent synthesizing final report...")
    final_response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    
    print("\n### Final Answer ###\n")
    print(final_response.choices[0].message.content)
else:
    # If the model didn't need tools, just print the direct answer
    print(response_message.content)
"""



'\n# Call and test the specified model\nprint("🤖 Agent analyzing query...")\nresponse = openai_client.chat.completions.create(\n    model="gpt-4o-mini",\n    messages=messages,\n    tools=tools,\n    tool_choice="auto"\n)\n\nresponse_message = response.choices[0].message\nmessages.append(response_message) # Append the model\'s response to keep history consistent\n\n# Step 5: Check if the model requested a tool call\nif response_message.tool_calls:\n    for tool_call in response_message.tool_calls:\n        if tool_call.function.name == "tavily_search":\n            # Extract arguments generated by the model\n            function_args = json.loads(tool_call.function.allocations or tool_call.function.arguments)\n            search_query = function_args.get("query")\n\n            print(f"🔍 Agent requested tool: tavily_search(query=\'{search_query}\')")\n\n            # Execute the actual web search using Tavily\n            tavily_response = tavily_client.search(query=search_query, max_r

## Initialize Tools

In [ ]:
import json
from tavily import TavilyClient
import arxiv

# Instantiate the Tavily client once so the LangGraph nodes can call it.
tavily_client = TavilyClient(api_key=tavily_api_key)

# Arxiv Search Function
def arxiv_search(query: str, max_results: int = 5) -> str:
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )
    
    results = []
    for result in search.results():
        paper_info = (
            f"Title: {result.title}\n"
            f"Authors: {', '.join(author.name for author in result.authors)}\n"
            f"Published: {result.published.strftime('%Y-%m-%d')}\n"
            f"URL: {result.pdf_url}\n"
            f"Abstract: {result.summary}\n"
            f"{'-'*40}"
        )
        results.append(paper_info)
        
    return "\n".join(results) if results else "No papers found for this query."

# 2. Map tool names to the actual functions
available_tools = {
    "arxiv_search": lambda query, max_results=5: arxiv_search(query=query, max_results=max_results),
    "tavily_search": lambda query, max_results=5: tavily_client.search(query=query, max_results=max_results)
}

# Tool Schemas

tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the live web for current facts, news, and technical data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The specific search query optimized for a search engine. Do not include punctuation or conversational filler."
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "arxiv_search",
            "description": "Search for academic papers, preprints, and research articles on arXiv.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The scientific topic or paper keywords to search for.",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "The number of research papers to return. Defaults to 3.",
                    },
                },
                "required": ["query"],
            },
        },
    }
]


## Create LangGraph State Class

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# State is one shared object every node reads from and writes to. 
# It is a TypedDict that defines the structure of the state object.
class AgentState(TypedDict):
    question: str
    answer: str
    threat_detected: bool
    messages: Annotated[list, add_messages]

## Define LangGraph Node Functions

In [ ]:
from typing import Dict, List

from langchain_core.messages import ToolMessage

# The node signatures below use `State`, an alias for the AgentState TypedDict
# defined in the "Create LangGraph State Class" section.
State = AgentState

# Web Search Result node: This node takes the current state, which includes a question, and uses the LLM to generate an answer. It returns a dictionary containing the answer.
def tavily_search_node(state: State) -> Dict[str, List[ToolMessage]]:
    """
    LangGraph Node that executes the Tavily search tool call.
    """
    latest_message = state["messages"][-1]
    tool_messages = []
    
    if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
        for tool_call in latest_message.tool_calls:
            if tool_call["name"] == "tavily_search":
                try:
                    # Execute Tavily search using arguments passed by the LLM
                    query = tool_call["args"].get("query")
                    tool_output = tavily_client.search(query=query)
                    
                    # Convert response dictionary/list to a JSON string
                    if isinstance(tool_output, (dict, list)):
                        tool_output = json.dumps(tool_output)
                        
                except Exception as e:
                    tool_output = f"Error executing Tavily search: {str(e)}"
                
                tool_messages.append(
                    ToolMessage(
                        content=str(tool_output),
                        tool_call_id=tool_call["id"],
                        name=tool_call["name"]
                    )
                )
                
    return {"messages": tool_messages}

# Arxiv Search node: This node takes the current state, which includes a question, 
# and uses the LLM to generate an answer. It returns a dictionary containing the answer.
def arxiv_search_node(state: State) -> Dict[str, List[ToolMessage]]:
    """
    LangGraph Node that executes the arXiv search tool call.
    """
    latest_message = state["messages"][-1]
    tool_messages = []
    
    if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
        for tool_call in latest_message.tool_calls:
            if tool_call["name"] == "arxiv_search":
                try:
                    # Extract arguments passed by the LLM
                    args = tool_call["args"]
                    query = args.get("query")
                    max_results = args.get("max_results", 5) # Default to 5 if not provided
                    
                    # Execute the local arXiv search function
                    tool_output = arxiv_search(query=query, max_results=max_results)
                        
                except Exception as e:
                    tool_output = f"Error executing arXiv search: {str(e)}"
                
                tool_messages.append(
                    ToolMessage(
                        content=str(tool_output),
                        tool_call_id=tool_call["id"],
                        name=tool_call["name"]
                    )
                )
                
    return {"messages": tool_messages}

## Define LangGraph Start Edge

In [ ]:
from uuid import uuid4

from langchain_core.messages import AIMessage

# Custom input for this run: edit the string, re-run this cell, then run the graph.
# Passing {"question": "..."} to app.invoke() overrides it for a single run.
CUSTOM_INPUT = "Recent advances in retrieval-augmented generation for scientific literature"


def start_node(state: AgentState) -> dict:
    """
    LangGraph START node.

    Turns the custom input into a single AIMessage holding one tool call per
    search tool, so the same question is handed to tavily_search_node and
    arxiv_search_node in parallel. Each node scans the latest message for the
    tool call carrying its own name.
    """
    question = (state.get("question") or CUSTOM_INPUT).strip()
    if not question:
        raise ValueError("CUSTOM_INPUT is empty - define the question for this run.")

    tool_calls = [
        {
            "name": "tavily_search",
            "args": {"query": question},
            "id": f"tavily-{uuid4().hex[:8]}",
            "type": "tool_call",
        },
        {
            "name": "arxiv_search",
            "args": {"query": question, "max_results": 5},
            "id": f"arxiv-{uuid4().hex[:8]}",
            "type": "tool_call",
        },
    ]

    return {
        "question": question,
        "messages": [AIMessage(content=question, tool_calls=tool_calls)],
    }


# Wire it up in the "Define LangGraph Edges" section:
# graph.add_node("start", start_node)
# graph.add_edge(START, "start")
# graph.add_edge("start", "tavily_search_node")
# graph.add_edge("start", "arxiv_search_node")


## Define LangGraph Security Guardrail Node

A conditional node placed between the start edge and the search nodes. It runs
`guardrails-ai-detect-prompt-injection` and `guardrails-ai-detect-jailbreak` on the
custom input. A flagged input is routed to a terminal node that reports the threat;
a clean input fans out to `tavily_search_node` and `arxiv_search_node`.

In [ ]:
import os

# Guardrails ships an OpenTelemetry exporter that retries an unreachable collector;
# disabling tracing keeps the notebook output clean.
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

from guardrails import Guard
from guardrails.settings import settings as guardrails_settings

guardrails_settings.disable_tracing = True

from guardrails_ai.detect_jailbreak import DetectJailbreak
from guardrails_ai.detect_prompt_injection import DetectPromptInjection
from langgraph.types import Command

# Security layer: both validators run on the custom input before any tool is called.
# Guard().use() stores one validator list per target, so both go in a single call.
# DetectJailbreak downloads its HuggingFace models the first time this cell runs.
# check_vector=False keeps the injection validator on its local heuristics: its
# optional embedding check would call OpenAI, and this notebook runs on Ollama
# with a placeholder OPENAI_API_KEY.
security_guard = Guard().use(
    DetectPromptInjection(check_vector=False, on_fail="noop"),
    DetectJailbreak(on_fail="noop"),
)

THREAT_MESSAGE = (
    "Request blocked: the input was flagged as a prompt-injection or jailbreak attempt, "
    "so no searches were run."
)


def security_check_node(state: AgentState) -> Command:
    """
    Conditional security node.

    Clean input fans out to both search nodes. A hostile input is routed to the
    terminal threat_detected node instead, so no tool ever runs.
    """
    question = state.get("question", "")
    outcome = security_guard.validate(question)

    if not outcome.validation_passed:
        return Command(goto="threat_detected", update={"threat_detected": True})

    return Command(
        goto=["tavily_search_node", "arxiv_search_node"],
        update={"threat_detected": False},
    )


def threat_detected_node(state: AgentState) -> dict:
    """Terminal node reached when the input is hostile: reports the block."""
    return {
        "answer": THREAT_MESSAGE,
        "messages": [AIMessage(content=THREAT_MESSAGE)],
    }


## Define LangGraph Edges

In [ ]:
from langgraph.graph import StateGraph, END, START

graph = StateGraph(AgentState)

# Nodes
graph.add_node("start", start_node)
graph.add_node("security", security_check_node)
graph.add_node("threat_detected", threat_detected_node)
graph.add_node("tavily_search_node", tavily_search_node)
graph.add_node("arxiv_search_node", arxiv_search_node)

# Start edge
graph.add_edge(START, "start")

# The custom input goes to the security layer next
graph.add_edge("start", "security")

# security_check_node returns a Command: a threat routes to "threat_detected",
# a clean input fans out to both search nodes in parallel.
graph.add_edge("threat_detected", END)
graph.add_edge("tavily_search_node", END)
graph.add_edge("arxiv_search_node", END)

app = graph.compile()
